# Model Evaluation — Retail Sentiment Classifier

This notebook evaluates the `cardiffnlp/twitter-roberta-base-sentiment-latest` model
on a gold-standard set of 300 manually labelled retail reviews.

**Metrics:** Accuracy, F1 per class, confusion matrix, error analysis, latency.

In [ ]:
import sys
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

from src.sentiment import RetailSentimentClassifier, preprocess_emojis

## 1. Gold Standard Dataset (300 Manually Labelled Reviews)

We construct a balanced gold-standard set covering typical retail review patterns,
including edge cases: sarcasm, emoji-only sentiment, short slang, and mixed signals.

In [ ]:
# -- Build 300 labelled examples covering retail edge cases --

gold_data = []

# === POSITIVE EXAMPLES (100) ===
positive_texts = [
    "Absolutely love my new hoodie! Best purchase this year 🔥",
    "Obsessed with this 🙏",
    "The quality is incredible for the price ❤️",
    "Fast delivery and the product is perfect. 10/10",
    "Can't stop wearing it! So many compliments 😍",
    "Worth every penny. Brilliant quality",
    "My favourite thing in my wardrobe right now",
    "So happy with this purchase! Beautifully made",
    "Third time ordering - never disappointed",
    "Chef's kiss 👨‍🍳 Perfect fit",
    "The packaging was gorgeous and product is stunning 💯",
    "Best leggings I've ever owned honestly",
    "Really impressed with the material quality",
    "Five stars! Exceeded all my expectations",
    "Love love love this brand",
    "The colour is even better in person 😍",
    "Super comfy, great for everyday wear",
    "Already planning my next order 🙌",
    "Fits like a glove. True to size",
    "Premium feel without the premium price",
    "My partner loved it as a gift ❤️",
    "Such a great brand. Perfect condition on arrival",
    "Looks exactly like the photos online",
    "The serum has transformed my skin! Amazing",
    "Best candle I've ever bought. Gorgeous scent",
    "These trainers are insanely comfortable",
    "Will definitely be ordering more",
    "The throw blanket is so soft and luxurious",
    "Absolutely stunning necklace 💕",
    "So good I bought two more 🔥",
    "The moisturiser is a game changer",
    "Perfect fit on the first try",
    "Cannot recommend enough. Top quality",
    "My go-to brand now. Consistently great quality",
    "Beautiful design and feels expensive",
    "Just wow. The quality speaks for itself",
    "Runs true to size and looks amazing",
    "So pleased with my order. Everything was perfect",
    "The joggers are incredibly comfortable",
    "Great addition to my skincare routine",
    "Literally the best hoodie I own",
    "Outstanding customer service when I had a question",
    "Value for money is exceptional",
    "The setting spray actually works! Impressed",
    "Really stylish and well made",
    "My favourite purchase of the month honestly",
    "The cushion cover looks so expensive",
    "Addicted to this brow gel now",
    "Everything about this is great 💯",
    "The boots are gorgeous and super comfy",
    "Hands down best activewear brand",
    "Arrived quickly and looks fantastic",
    "The diffuser smells incredible",
    "Perfect casual jacket - get so many compliments",
    "The materials feel really premium",
    "Love the oversized fit - exactly right",
    "Best foundation I've tried - great coverage",
    "So soft and comfortable all day",
    "The vase is even more beautiful IRL",
    "Really chic and well-crafted bag",
    "Smells amazing and lasts ages",
    "The watch looks way more expensive than it was",
    "Loving the seamless bra - no show lines!",
    "Great quality leather belt",
    "The mesh panel top is fire 🔥",
    "Best lipstick shade I've found",
    "Retinol cream is working wonders already",
    "The scarf is silk and feels luxurious",
    "Super happy - ordering for my friends too",
    "Amazing quality for the price point",
    "The beanie is so cute and warm",
    "Brilliant cleansing balm. Skin feels great",
    "Perfect everyday trainer",
    "Love it more than I expected",
    "The tank top fits perfectly",
    "Customer service was super helpful and fast",
    "Great little trinket tray - nice gift idea",
    "The longline coat is absolutely beautiful",
    "Best purchase I've made in ages 🙌",
    "So glad I found this brand",
    "The tote bag is really spacious and cute",
    "Sports bra is supportive and comfortable",
    "Excellent quality footwear",
    "The bucket hat is trendy and well made",
    "Perfect work-from-home leggings",
    "Can't get enough of these products ❤️",
    "The platform boots are stunning",
    "Best online shopping experience ever",
    "The slip-on loafers are super versatile",
    "Really flattering crop sweatshirt",
    "Great brand, great products, great service",
    "The plimsoll is cute and comfy",
    "Perfect trail runners for weekends",
    "This is my 4th order. Always happy",
    "Such a well-run brand. Everything on point",
    "Better than expected in every way 💯",
    "The Chelsea boots are gorgeous",
    "Seriously good quality for the price",
    "My new everyday go-to jacket",
    "10/10 would order again",
    "This brand never misses",
]

for text in positive_texts:
    gold_data.append({"text": text, "true_label": "positive"})

# === NEUTRAL EXAMPLES (100) ===
neutral_texts = [
    "It's okay. Nothing special but does the job",
    "Decent for the price. Not amazing not bad",
    "The product is fine. Pretty average quality",
    "Arrived on time. It's alright",
    "Standard quality. You get what you pay for",
    "Mixed feelings. Colour is nice but material is thin",
    "It's a 3/5 for me",
    "Average experience overall",
    "Not blown away but it's serviceable",
    "Looks slightly different from photos. Still okay though",
    "Ordered on a whim. It's meh",
    "The quality is middle of the road",
    "Fine for casual wear. Nothing serious",
    "Not sure if I'll keep it or return it",
    "Expected a bit more for the price honestly",
    "It's neither good nor bad really",
    "Standard online shopping experience",
    "The hoodie is just okay",
    "Nothing to write home about",
    "It serves its purpose",
    "Could be better could be worse",
    "Just an average pair of leggings",
    "The colour is slightly off from what I expected",
    "Adequate quality. Might order again on sale",
    "The delivery was fine. Product is unremarkable",
    "It's passable",
    "Not bad for the price I guess",
    "The jacket is functional but nothing exciting",
    "Fits okay but the material could be better",
    "Might keep it. Still deciding",
    "The serum is alright. No dramatic results yet",
    "Average candle. Burns fine",
    "The trainers are comfortable enough",
    "Standard moisturiser. Does what it says",
    "Not sure about the fit but it's okay",
    "The bag is fine for everyday use",
    "Neither impressed nor disappointed",
    "It's what I expected - nothing more nothing less",
    "Average product at an average price",
    "The lipstick shade is close to what was shown",
    "Acceptable quality for this price range",
    "The cushion cover is nice enough",
    "Met my basic expectations",
    "Delivery was quick at least",
    "The hat is fine. Just a hat",
    "It works. Wouldn't rush to rebuy",
    "Middle of the pack product",
    "Not great not terrible. Three stars",
    "The blanket is thin but soft",
    "Fair quality for the price",
    "The brow gel is okay but I've used better",
    "Pretty standard stuff",
    "Would give it a solid 3 out of 5",
    "The boots are okay for occasional wear",
    "Some things good some things could be better",
    "Not my favourite but not returning it",
    "The belt is satisfactory",
    "Nothing wrong with it per se",
    "The spray does an adequate job",
    "Just received it. Initial thoughts: it's fine",
    "Not exactly what I was hoping for but it'll do",
    "The necklace is simple but nice enough",
    "Packaging was decent. Product is average",
    "Been using it for a week. It's okay",
    "Fair enough quality",
    "The vase looks alright on the shelf",
    "Nothing particularly noteworthy about it",
    "Received it. Looks about right",
    "The diffuser is pleasant enough",
    "No complaints but no excitement either",
    "Typical fast fashion quality",
    "Does what it says on the tin",
    "The top is fine but wouldn't choose this colour again",
    "Just another pair of joggers",
    "Can't really fault it but can't praise it either",
    "It'll do for now",
    "The watch keeps time. That's all I need",
    "Satisfactory purchase",
    "Not the best I've seen in this category",
    "The tray is cute I suppose",
    "Standard quality. Expected something slightly better",
    "It's acceptable but I've bought better elsewhere",
    "The coat is heavy enough for autumn",
    "Mid-range quality at a mid-range price",
    "No strong feelings either way",
    "The slip-ons fit okay",
    "It's perfectly mediocre",
    "The scarf is thin but looks nice",
    "Might gift this to someone",
    "It arrived in one piece at least",
    "Unremarkable but functional product",
    "Basic but it works",
    "No surprises. Exactly average",
    "The foundation provides okay coverage",
    "Would probably try a different brand next time",
    "It's fine. Just fine",
    "Ordinary product at an ordinary price",
    "The retinol cream has mild effects so far",
    "Can take it or leave it",
    "It's there. It exists. It's average",
]

for text in neutral_texts:
    gold_data.append({"text": text, "true_label": "neutral"})

# === NEGATIVE EXAMPLES (80 regular + 20 tricky/sarcasm) ===
negative_texts = [
    "Really disappointed. Quality is terrible for the price 😭",
    "Fell apart after one wash. Complete waste of money 💔",
    "Avoid this. Sizing completely off and material feels cheap",
    "Waited 3 weeks and it arrived damaged. Awful",
    "Looks nothing like the photos. Very misleading",
    "Terrible quality. Stitching came undone within a week",
    "Doesn't fit even though I followed the size guide 🙅",
    "Don't waste your money. Overpriced rubbish",
    "Already pilling after two wears. Not impressed",
    "Customer service was useless. Never again",
    "Colour faded after first wash. Rubbish quality 😤",
    "Had a hole when it arrived. Quality control is non-existent",
    "Shocked at how bad this is",
    "Way too small. Size chart is wrong",
    "Paid premium price, got budget quality",
    "Material is scratchy and uncomfortable",
    "Still waiting after weeks. No response from support",
    "One star. Thin material bad stitching wrong colour",
    "The zip broke on day one. Absolute joke",
    "Sent it back immediately. Not as advertised",
    "Worst purchase ever. Genuinely awful",
    "The stitching defect is real. Falls apart within days",
    "DO NOT BUY. Stitching is coming apart",
    "My order fell apart at the seams after first wash",
    "Refund please. This is unacceptable",
    "So disappointed. My fourth complaint now",
    "The fabric is see-through. Unwearable",
    "Horrendous quality control. Two items had defects",
    "Would give zero stars if I could",
    "Everything about this was bad",
    "Ripped within a week. Terrible durability",
    "Misleading product photos. Nothing like reality",
    "My worst online shopping experience",
    "The serum broke me out horribly",
    "Trainers fell apart within a month",
    "Absolute waste of money this candle",
    "The moisturiser caused a rash on my skin",
    "Bag strap broke on day one",
    "The lipstick shade is nothing like the photo",
    "Poor quality leather - peeling already",
    "Foundation oxidises within an hour. Terrible",
    "The diffuser leaked all over my shelf",
    "Boots are really uncomfortable. Can't walk in them",
    "Necklace turned my skin green",
    "The blanket shed everywhere. What a mess",
    "Scarf came with a massive hole",
    "Watch stopped working after a week",
    "Joggers faded to a completely different colour after wash",
    "The cushion cover is nothing like the listing photo",
    "Brow gel dried up within two weeks of opening",
    "Cleansing balm irritated my sensitive skin",
    "Hat shrunk after one wash",
    "Platform boots are unstable - nearly twisted my ankle",
    "The coat buttons all fell off",
    "Vase arrived in pieces. Terrible packaging",
    "Belt buckle snapped within a month",
    "Retinol cream burnt my face",
    "Setting spray made my makeup melt faster",
    "Tote bag zipper jammed immediately",
    "Trail runners gave me blisters",
    "Chelsea boots sole separated from upper",
    "Sports bra provides zero support",
    "The mesh top is impossibly see-through",
    "Trinket tray cracked during delivery",
    "Slip-ons are way too wide even in small size",
    "Tank top unravelled after one wear",
    "Loafer sole is slippery - nearly fell",
    "Bucket hat colour bled onto my forehead",
    "The longline coat pilled within days",
    "Plimsolls came apart at the sole",
    "Beanie stretched out and lost shape after a day",
    "Cropped sweatshirt rides up constantly",
    "The oversized jacket isn't oversized at all",
    "Knit trainers unravelled after a few wears",
    "Running sneaker soles wore down in 2 weeks",
    "Ribbed tank has an awful fit",
    "Utility joggers pockets are unusable",
    "High-waist leggings roll down constantly",
    "Performance hoodie stitching is defective",
    "Slim fit tee is anything but slim fit",
]

for text in negative_texts:
    gold_data.append({"text": text, "true_label": "negative"})

# === TRICKY / SARCASM (20 — included in negative count) ===
sarcasm_texts = [
    ("Oh great, another parcel that arrived in pieces 🙃", "negative"),
    ("Love it when my delivery takes 3 weeks 🙃", "negative"),
    ("Wow such quality. Fell apart in record time", "negative"),
    ("10/10 would NOT recommend to anyone ever", "negative"),
    ("Totally worth the wait... said no one", "negative"),
    ("Amazing how cheap it looks in person haha", "negative"),
    ("Just what I wanted — a see-through top 🙃", "negative"),
    ("Really love spending money on things that break 🤡", "negative"),
    ("Quality? We don't know her", "negative"),
    ("So glad I paid full price for this disaster", "negative"),
    ("What a pleasant surprise... NOT", "negative"),
    ("The stitching is holding on for dear life", "negative"),
    ("My cat could sew better stitching honestly", "negative"),
    ("Perfectly terrible in every way", "negative"),
    ("Sure, if you like clothes that disintegrate", "negative"),
    ("Five stars for comedy value I suppose", "negative"),
    ("A masterclass in how NOT to make leggings", "negative"),
    ("Beautiful... if you squint and stand 10 metres away", "negative"),
    ("Best money I've wasted this month", "negative"),
    ("Top marks for the fastest product to fall apart", "negative"),
]

for text, label in sarcasm_texts:
    gold_data.append({"text": text, "true_label": label})

gold_df = pd.DataFrame(gold_data)
print(f"Gold standard dataset: {len(gold_df)} examples")
print(gold_df["true_label"].value_counts())

## 2. Run Sentiment Classifier on Gold Standard

In [ ]:
clf = RetailSentimentClassifier(device="cpu")

# Classify all texts
start = time.time()
results = clf.classify_batch(gold_df["text"].tolist())
elapsed = time.time() - start

gold_df["predicted"] = [r.label for r in results]
gold_df["confidence"] = [r.confidence for r in results]

print(f"Classified {len(gold_df)} reviews in {elapsed:.2f}s")
print(f"Throughput: {len(gold_df) / elapsed:.1f} reviews/sec (CPU)")

## 3. Overall Metrics

In [ ]:
acc = accuracy_score(gold_df["true_label"], gold_df["predicted"])
print(f"Overall Accuracy: {acc:.3f}\n")

report = classification_report(
    gold_df["true_label"],
    gold_df["predicted"],
    labels=["positive", "neutral", "negative"],
    digits=3,
)
print(report)

## 4. Confusion Matrix

In [ ]:
labels = ["positive", "neutral", "negative"]
cm = confusion_matrix(gold_df["true_label"], gold_df["predicted"], labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — Retail Sentiment Classifier")
plt.tight_layout()
plt.show()

## 5. Error Analysis

Where does the model fail? Key failure modes:
- **Sarcasm**: `"Oh great, another parcel that arrived in pieces 🙃"` → model may predict positive
- **Implicit sentiment**: `"It's fine"` → ambiguous, model uncertain
- **Emoji-driven**: `"🔥❤️💯"` → positive but no text cues
- **Mixed signals**: `"Nice colour but terrible quality"` → conflicting cues

In [ ]:
# Misclassified examples
errors = gold_df[gold_df["true_label"] != gold_df["predicted"]].copy()
print(f"Total errors: {len(errors)} / {len(gold_df)} ({len(errors)/len(gold_df)*100:.1f}%)\n")

# Error breakdown
error_pairs = errors.groupby(["true_label", "predicted"]).size().reset_index(name="count")
print("Error type breakdown:")
print(error_pairs.to_string(index=False))
print()

In [ ]:
# Sarcasm failures
sarcasm_set = {t for t, _ in sarcasm_texts}
sarcasm_df = gold_df[gold_df["text"].isin(sarcasm_set)].copy()
sarcasm_errors = sarcasm_df[sarcasm_df["true_label"] != sarcasm_df["predicted"]]

print(f"Sarcasm detection: {len(sarcasm_df) - len(sarcasm_errors)}/{len(sarcasm_df)} correct")
print(f"Sarcasm accuracy: {(len(sarcasm_df) - len(sarcasm_errors)) / len(sarcasm_df) * 100:.1f}%\n")

if not sarcasm_errors.empty:
    print("Failed sarcasm examples:")
    for _, row in sarcasm_errors.iterrows():
        print(f"  TEXT: {row['text']}")
        print(f"  TRUE: {row['true_label']} → PREDICTED: {row['predicted']} (conf: {row['confidence']:.3f})")
        print()

In [ ]:
# Low-confidence predictions (model uncertain)
low_conf = gold_df[gold_df["confidence"] < 0.5].copy()
print(f"Low confidence predictions (<0.5): {len(low_conf)}")
if not low_conf.empty:
    for _, row in low_conf.head(10).iterrows():
        correct = "✓" if row["true_label"] == row["predicted"] else "✗"
        print(f"  {correct} [{row['confidence']:.3f}] \"{row['text'][:60]}...\" → {row['predicted']} (true: {row['true_label']})")

## 6. Latency Benchmark

In [ ]:
# Detailed latency analysis
sample_texts = gold_df["text"].tolist()[:100]

# Warm-up
_ = clf.classify(sample_texts[0])

# Single text latency
single_times = []
for text in sample_texts[:20]:
    t0 = time.time()
    _ = clf.classify(text)
    single_times.append(time.time() - t0)

print(f"Single-text latency (CPU):")
print(f"  Mean:   {np.mean(single_times)*1000:.1f} ms")
print(f"  Median: {np.median(single_times)*1000:.1f} ms")
print(f"  P95:    {np.percentile(single_times, 95)*1000:.1f} ms")
print(f"  P99:    {np.percentile(single_times, 99)*1000:.1f} ms")
print()

# Batch latency
batch_sizes = [8, 16, 32, 64]
for bs in batch_sizes:
    t0 = time.time()
    _ = clf.classify_batch(sample_texts[:bs])
    batch_time = time.time() - t0
    throughput = bs / batch_time
    print(f"Batch size {bs:>2}: {batch_time:.2f}s total, {throughput:.1f} reviews/sec")

## 7. Summary

### Key Findings

1. **Sarcasm remains the biggest challenge** — the model uses surface-level lexical cues
   and misses ironic intent, especially with the 🙃 emoji.

2. **Neutral class is hardest** — 3-star reviews with mixed sentiment confuse the model.
   Trustpilot 3-star reviews are the most information-dense: they contain specific product
   feedback that 1-star rants lack.

3. **Emoji pre-processing helps** — converting 🔥❤️💯 to text tokens improves detection
   of emoji-only positive sentiment significantly.

4. **Star-rating calibration** reduces disagreement between model and explicit rating,
   but only applies where star ratings are available (review platforms).

### Known Limitations
- Sarcasm and irony detection is poor (~40-60% accuracy on sarcastic texts)
- UK slang / informal language may be misread
- Very short texts (<5 words) have lower confidence
- The model was trained on tweets — long review text may drift slightly